# Statistical analysis: five-fold stress test, McNemar tests and bootstrap CIs
**What this notebook does**
1. Rebuilds the exact 991-image dataset and the same five folds as the main run (seed 42).
2. Trains each of the 5 CNNs on each fold (same settings as the main run) and saves **every image's prediction**
   on five versions of the validation images: original, JPEG q90, resized 256 × 256, grey background, harmonised (all three).
3. Pools the out-of-fold predictions (all 991 images, each predicted by a model that never saw it) and computes:
   - accuracy, healthy recall, unhealthy recall with **95% Clopper–Pearson CIs**
   - **95% bootstrap CIs** for accuracy, F1 and AUC (2,000 resamples)
   - **exact McNemar tests** within each model (original vs each altered version)
   - **exact McNemar tests between models** (all pairs, on original and harmonised images; Holm-corrected)
   - **Cochran's Q test** (do the five models differ at all?) on original and harmonised images
4. Saves paper-ready tables and a forest-plot figure to `OUT_DIR/stats/`.

**Runtime:** about 1.5–2.5 h on a T4 GPU (25 training runs). It **resumes automatically** – if Colab disconnects,
just run all cells again and finished model/fold runs are skipped.

## 1. Setup – edit the two paths

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, io, copy, json, random, itertools, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.stats import beta, binom, chi2

DATA_DIR = os.environ.get("DATA_DIR", "/content/drive/MyDrive/Citrus_Plant_Leaves")   # <- EDIT if different
OUT_DIR  = os.environ.get("OUT_DIR",  "/content/drive/MyDrive/citrus_results")   # <- EDIT: folder with duplicates_removed.csv
QUICK_TEST = os.environ.get("QUICK_TEST", "0") == "1"
PRETRAINED = os.environ.get("PRETRAINED", "1") == "1"
MODEL_NAMES = ["mobilenet_v2", "efficientnet_b0", "shufflenet_v2", "resnet18", "squeezenet1_1"]
PRETTY = {"mobilenet_v2": "MobileNetV2", "efficientnet_b0": "EfficientNet-B0", "shufflenet_v2": "ShuffleNetV2 1.0×",
          "resnet18": "ResNet-18", "squeezenet1_1": "SqueezeNet 1.1"}
SEED, IMG, BATCH, LR = 42, 224, (16 if QUICK_TEST else 32), 1e-4
EPOCHS, PATIENCE = (2, 1) if QUICK_TEST else (20, 5)
N_FOLDS = 2 if QUICK_TEST else 5
N_BOOT = 200 if QUICK_TEST else 2000
STATS_DIR = os.path.join(OUT_DIR, "stats"); PRED_DIR = os.path.join(STATS_DIR, "predictions")
os.makedirs(PRED_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

## 2. Rebuild the 991-image list and the five folds of the main run

In [ ]:
EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")
dup_csv = os.path.join(OUT_DIR, "duplicates_removed.csv")
assert os.path.exists(dup_csv), f"Not found: {dup_csv} – set OUT_DIR to the main-run results folder"
class_dirs = {}
for d, _, files in os.walk(DATA_DIR):
    if any(f.lower().endswith(EXTS) for f in files): class_dirs[os.path.basename(d)] = d
assert len(class_dirs) == 2, f"Expected 2 image folders, found {list(class_dirs)}"
CLASSES = sorted(class_dirs, key=lambda c: (0 if c.lower().startswith("healthy") else 1, c))
removed = {(os.path.basename(os.path.dirname(p)), os.path.basename(p)) for p in pd.read_csv(dup_csv)["removed"]}
paths, labels = [], []
for i, c in enumerate(CLASSES):
    for f in sorted(os.listdir(class_dirs[c])):
        if f.lower().endswith(EXTS) and (c, f) not in removed:
            paths.append(os.path.join(class_dirs[c], f)); labels.append(i)
labels = np.array(labels)
info_path = os.path.join(OUT_DIR, "run_info.json")
if os.path.exists(info_path) and not QUICK_TEST:
    assert len(paths) == json.load(open(info_path))["n_images"], "Image count differs from the main run – check paths"
folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(paths, labels))
print(len(paths), "images", dict(zip(CLASSES, np.bincount(labels))), "|", N_FOLDS, "folds")

## 3. Transformations, models and the five image versions

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([transforms.RandomResizedCrop(IMG, scale=(0.8, 1.0)), transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(), transforms.RandomRotation(15), transforms.ColorJitter(0.2, 0.2, 0.1),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([transforms.Resize((IMG, IMG)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class DS(Dataset):
    def __init__(s, items, y, tf): s.items, s.y, s.tf = items, y, tf
    def __len__(s): return len(s.items)
    def __getitem__(s, i):
        im = s.items[i] if isinstance(s.items[i], Image.Image) else Image.open(s.items[i]).convert("RGB")
        return s.tf(im), int(s.y[i])

def build_model(name):
    W = "DEFAULT" if PRETRAINED else None
    if name == "mobilenet_v2":    m = models.mobilenet_v2(weights=W); m.classifier[1] = nn.Linear(m.last_channel, 2)
    elif name == "efficientnet_b0": m = models.efficientnet_b0(weights=W); m.classifier[1] = nn.Linear(m.classifier[1].in_features, 2)
    elif name == "shufflenet_v2": m = models.shufflenet_v2_x1_0(weights=W); m.fc = nn.Linear(m.fc.in_features, 2)
    elif name == "resnet18":      m = models.resnet18(weights=W); m.fc = nn.Linear(m.fc.in_features, 2)
    elif name == "squeezenet1_1": m = models.squeezenet1_1(weights=W); m.classifier[1] = nn.Conv2d(512, 2, 1)
    return m

def leaf_mask(im):
    s = np.asarray(im.convert("HSV"))[:, :, 1].astype(np.uint8)
    hist = np.bincount(s.ravel(), minlength=256).astype(float); w = np.cumsum(hist); mu = np.cumsum(hist * np.arange(256))
    with np.errstate(divide="ignore", invalid="ignore"):
        between = (mu[-1] * w / w[-1] - mu) ** 2 / (w * (w[-1] - w))
    return s > int(np.nanargmax(between[:-1]))
def to_jpeg(im, q=90):
    b = io.BytesIO(); im.convert("RGB").save(b, "JPEG", quality=q); b.seek(0); return Image.open(b).convert("RGB")
def grey_bg(im):
    a = np.asarray(im.convert("RGB")).copy(); a[~leaf_mask(im)] = (128, 128, 128); return Image.fromarray(a)
def to_256(im): return im.convert("RGB").resize((256, 256), Image.BILINEAR)
VARIANTS = {"original": lambda im: im, "jpeg_q90": to_jpeg, "resized_256": to_256,
            "grey_background": grey_bg, "harmonised": lambda im: grey_bg(to_jpeg(to_256(im)))}
VARIANT_PRETTY = {"original": "Original", "jpeg_q90": "JPEG re-encoded", "resized_256": "Resized 256 × 256",
                  "grey_background": "Grey background", "harmonised": "Harmonised"}

## 4. Train every model on every fold and save per-image predictions (resumable)

In [ ]:
def predict(model, loader):
    model.eval(); ps = []
    with torch.no_grad():
        for x, _ in loader: ps.append(torch.softmax(model(x.to(DEVICE)), 1)[:, 1].cpu().numpy())
    return np.concatenate(ps)

def train_fold(name, tr, va, k):
    set_seed(SEED + k)
    nw = 0 if QUICK_TEST else 2
    tr_dl = DataLoader(DS([paths[i] for i in tr], labels[tr], train_tf), BATCH, shuffle=True, num_workers=nw)
    va_dl = DataLoader(DS([paths[i] for i in va], labels[va], eval_tf), BATCH, num_workers=nw)
    m = build_model(name).to(DEVICE)
    w = torch.tensor(len(tr) / (2 * np.bincount(labels[tr], minlength=2)), dtype=torch.float32, device=DEVICE)
    crit, opt = nn.CrossEntropyLoss(weight=w), torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    best, state, wait = 1e9, copy.deepcopy(m.state_dict()), 0
    for ep in range(EPOCHS):
        m.train()
        for x, y in tr_dl:
            opt.zero_grad(); loss = crit(m(x.to(DEVICE)), y.to(DEVICE)); loss.backward(); opt.step()
        m.eval(); vl = 0
        with torch.no_grad():
            for x, y in va_dl: vl += crit(m(x.to(DEVICE)), y.to(DEVICE)).item() * len(y)
        vl /= len(va)
        if vl < best: best, state, wait = vl, copy.deepcopy(m.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE: break
    m.load_state_dict(state); return m

t0 = time.time()
for k, (tr, va) in enumerate(folds):
    todo = [n for n in MODEL_NAMES if not os.path.exists(os.path.join(PRED_DIR, f"{n}_fold{k+1}.csv"))]
    if not todo: print(f"fold {k+1}: all models done – skipped"); continue
    # build the five versions of this fold's validation images once
    base = [Image.open(paths[i]).convert("RGB") for i in va]
    versions = {v: [fn(im) for im in base] for v, fn in VARIANTS.items()}
    for name in todo:
        m = train_fold(name, tr, va, k)
        rows = {"image": [os.path.basename(paths[i]) for i in va], "path": [paths[i] for i in va],
                "label": labels[va], "fold": k + 1, "model": name}
        df = pd.DataFrame(rows)
        for v, imgs in versions.items():
            df[f"prob_{v}"] = predict(m, DataLoader(DS(imgs, labels[va], eval_tf), BATCH))
        df.to_csv(os.path.join(PRED_DIR, f"{name}_fold{k+1}.csv"), index=False)
        acc = ((df.prob_original >= .5) == df.label).mean(); acc_h = ((df.prob_harmonised >= .5) == df.label).mean()
        print(f"fold {k+1} {PRETTY[name]:18s} original {acc*100:6.2f}%  harmonised {acc_h*100:6.2f}%   ({(time.time()-t0)/60:.0f} min)")
        del m; torch.cuda.empty_cache()

pred = pd.concat([pd.read_csv(os.path.join(PRED_DIR, f"{n}_fold{k+1}.csv")) for n in MODEL_NAMES for k in range(N_FOLDS)])
pred.to_csv(os.path.join(STATS_DIR, "per_image_predictions.csv"), index=False)
print("Saved per-image predictions:", pred.shape)

## 5. Statistical helper functions

In [ ]:
def clopper_pearson(k, n, a=0.05):
    lo = 0.0 if k == 0 else beta.ppf(a / 2, k, n - k + 1)
    hi = 1.0 if k == n else beta.ppf(1 - a / 2, k + 1, n - k)
    return lo * 100, hi * 100

def mcnemar_exact(correct_a, correct_b):
    """Exact two-sided McNemar test on paired correct/incorrect vectors."""
    b = int(np.sum(correct_a & ~correct_b)); c = int(np.sum(~correct_a & correct_b))
    p = 1.0 if b + c == 0 else min(1.0, 2 * binom.cdf(min(b, c), b + c, 0.5))
    return b, c, p

def holm(pvals):
    p = np.asarray(pvals); order = np.argsort(p); adj = np.empty_like(p); run = 0
    for rank, i in enumerate(order):
        run = max(run, (len(p) - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

def cochrans_q(correct_matrix):
    """correct_matrix: n_images x k_models (0/1). Returns Q, df, p."""
    X = np.asarray(correct_matrix, dtype=float); k = X.shape[1]
    C = X.sum(0); R = X.sum(1); N = X.sum()
    denom = k * N - np.sum(R ** 2)
    if denom == 0: return 0.0, k - 1, 1.0
    Q = (k - 1) * (k * np.sum(C ** 2) - N ** 2) / denom
    return Q, k - 1, chi2.sf(Q, k - 1)

rng = np.random.default_rng(SEED)
def bootstrap_ci(y, p, n_boot=N_BOOT):
    n = len(y); accs, f1s, aucs = [], [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n); yy, pp = y[idx], p[idx]; yh = (pp >= .5).astype(int)
        accs.append(accuracy_score(yy, yh)); f1s.append(f1_score(yy, yh, zero_division=0))
        if len(np.unique(yy)) == 2: aucs.append(roc_auc_score(yy, pp))
    q = lambda v: (np.percentile(v, 2.5) * 100, np.percentile(v, 97.5) * 100)
    return q(accs), q(f1s), q(aucs) if aucs else (np.nan, np.nan)

## 6. Table A – accuracy and per-class recall with 95% exact CIs; bootstrap CIs for F1 and AUC (pooled, all images)

In [ ]:
rowsA = []
for name in MODEL_NAMES:
    d = pred[pred.model == name].sort_values("path"); y = d.label.values
    for v in VARIANTS:
        p = d[f"prob_{v}"].values; yh = (p >= .5).astype(int); corr = yh == y
        n, k = len(y), int(corr.sum())
        hk, hn = int(((yh == 0) & (y == 0)).sum()), int((y == 0).sum())
        uk, un = int(((yh == 1) & (y == 1)).sum()), int((y == 1).sum())
        (bal, bah), (f1l, f1h), (aul, auh) = bootstrap_ci(y, p)
        rowsA.append({"Model": PRETTY[name], "Images": VARIANT_PRETTY[v],
            "Accuracy (%)": round(k / n * 100, 2), "Accuracy 95% CI": f"{clopper_pearson(k, n)[0]:.2f}–{clopper_pearson(k, n)[1]:.2f}",
            "Healthy recall (%)": round(hk / hn * 100, 2), "Healthy recall 95% CI": f"{clopper_pearson(hk, hn)[0]:.1f}–{clopper_pearson(hk, hn)[1]:.1f}",
            "Unhealthy recall (%)": round(uk / un * 100, 2), "Unhealthy recall 95% CI": f"{clopper_pearson(uk, un)[0]:.1f}–{clopper_pearson(uk, un)[1]:.1f}",
            "F1 (%)": round(f1_score(y, yh, zero_division=0) * 100, 2), "F1 95% bootstrap CI": f"{f1l:.2f}–{f1h:.2f}",
            "AUC (%)": round(roc_auc_score(y, p) * 100, 2), "AUC 95% bootstrap CI": f"{aul:.2f}–{auh:.2f}",
            "_k": k, "_n": n})
tabA = pd.DataFrame(rowsA)
tabA.drop(columns=["_k", "_n"]).to_csv(os.path.join(STATS_DIR, "tableA_accuracy_CI_bootstrap.csv"), index=False)
tabA.drop(columns=["_k", "_n"])

## 7. Table B – McNemar within each model: original vs each altered version
`b` = images correct on original but wrong on the altered version; `c` = the reverse.

In [ ]:
rowsB = []
for name in MODEL_NAMES:
    d = pred[pred.model == name].sort_values("path"); y = d.label.values
    c0 = (d.prob_original.values >= .5).astype(int) == y
    for v in [v for v in VARIANTS if v != "original"]:
        cv = (d[f"prob_{v}"].values >= .5).astype(int) == y
        b, c, p = mcnemar_exact(c0, cv)
        rowsB.append({"Model": PRETTY[name], "Comparison": f"Original vs {VARIANT_PRETTY[v]}",
                      "Accuracy original (%)": round(c0.mean() * 100, 2), "Accuracy altered (%)": round(cv.mean() * 100, 2),
                      "b (lost)": b, "c (gained)": c, "p (exact)": p})
tabB = pd.DataFrame(rowsB); tabB["p (Holm)"] = holm(tabB["p (exact)"])
tabB.to_csv(os.path.join(STATS_DIR, "tableB_mcnemar_within_model.csv"), index=False)
tabB.style.format({"p (exact)": "{:.2e}", "p (Holm)": "{:.2e}"})

## 8. Table C – McNemar between models (all 10 pairs) on original and harmonised images, Holm-corrected

In [ ]:
rowsC = []
for v in ["original", "harmonised"]:
    corr = {}
    for name in MODEL_NAMES:
        d = pred[pred.model == name].sort_values("path")
        corr[name] = (d[f"prob_{v}"].values >= .5).astype(int) == d.label.values
    block = []
    for a, b_ in itertools.combinations(MODEL_NAMES, 2):
        b, c, p = mcnemar_exact(corr[a], corr[b_])
        block.append({"Images": VARIANT_PRETTY[v], "Model A": PRETTY[a], "Model B": PRETTY[b_],
                      "Accuracy A (%)": round(corr[a].mean() * 100, 2), "Accuracy B (%)": round(corr[b_].mean() * 100, 2),
                      "A right, B wrong": b, "A wrong, B right": c, "p (exact)": p})
    adj = holm([r["p (exact)"] for r in block])
    for r, pa in zip(block, adj): r["p (Holm)"] = pa
    rowsC += block
tabC = pd.DataFrame(rowsC)
tabC.to_csv(os.path.join(STATS_DIR, "tableC_mcnemar_between_models.csv"), index=False)
tabC.style.format({"p (exact)": "{:.2e}", "p (Holm)": "{:.2e}"})

## 9. Table D – Cochran's Q: do the five models differ at all?

In [ ]:
rowsD = []
for v in VARIANTS:
    M = np.column_stack([((pred[pred.model == n].sort_values("path")[f"prob_{v}"].values >= .5).astype(int)
                          == pred[pred.model == n].sort_values("path").label.values) for n in MODEL_NAMES])
    Q, df_, p = cochrans_q(M)
    rowsD.append({"Images": VARIANT_PRETTY[v], "Cochran's Q": round(Q, 2), "df": df_, "p": p})
tabD = pd.DataFrame(rowsD); tabD.to_csv(os.path.join(STATS_DIR, "tableD_cochrans_q.csv"), index=False)
tabD.style.format({"p": "{:.2e}"})

## 10. Figure – accuracy with 95% CIs (forest plot) for the paper

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.6))
vs = list(VARIANTS); offs = np.linspace(-0.3, 0.3, len(vs)); colors = plt.cm.viridis(np.linspace(0, 0.85, len(vs)))
for j, v in enumerate(vs):
    sub = tabA[tabA.Images == VARIANT_PRETTY[v]]
    for i, name in enumerate(MODEL_NAMES):
        r = sub[sub.Model == PRETTY[name]].iloc[0]; lo, hi = clopper_pearson(r["_k"], r["_n"])
        ax.errorbar(r["Accuracy (%)"], i + offs[j], xerr=[[r["Accuracy (%)"] - lo], [hi - r["Accuracy (%)"]]],
                    fmt="o", color=colors[j], ms=4, capsize=2, label=VARIANT_PRETTY[v] if i == 0 else None)
ax.set_yticks(range(len(MODEL_NAMES))); ax.set_yticklabels([PRETTY[n] for n in MODEL_NAMES]); ax.invert_yaxis()
ax.set_xlabel(f"Accuracy (%) with 95% Clopper–Pearson CI (pooled out-of-fold, n = {len(paths)})")
ax.grid(axis="x", alpha=.3); ax.legend(fontsize=8, loc="lower left", frameon=False)
plt.tight_layout(); plt.savefig(os.path.join(STATS_DIR, "fig_forest_accuracy_CI.png"), dpi=300); plt.show()

## 11. Summary to send back
Send the whole `stats` folder **except** the `predictions` sub-folder (or zip it all – it is small).

In [ ]:
print("Cochran's Q (original):", tabD.iloc[0].to_dict())
print("Cochran's Q (harmonised):", tabD.iloc[-1].to_dict())
print("\nFiles in", STATS_DIR)
for f in sorted(os.listdir(STATS_DIR)): print("  ", f)